In [ ]:
# Token Lexer
from enum import Enum

class TOKEN_TYPE(Enum):
    OR = 1
    OPEN_PARENTHESIS = 2
    CLOSED_PARENTHESIS = 3
    OPEN_SQUARE_BRACKET = 4
    CLOSED_SQUARE_BRACKET = 5
    DASH = 6
    DOT = 7
    QUESTION_MARK = 8
    STAR = 9
    PLUS = 10
    LITERAL_CHARACTER = 11

class token():
    def __init__(self, token_type: TOKEN_TYPE, value: chr):
        self.token_type = token_type
        self.value = value
    def __repr__(self):
        return f"<{self.token_type}: {self.value}>"

def tokenize(regex_string) -> list[token]:
    tokens:list[token] = []
    # These are the special characters your lexer needs to recognize
    meta_chars = {
        '|': TOKEN_TYPE.OR,
        '*': TOKEN_TYPE.STAR,
        '+': TOKEN_TYPE.PLUS,
        '.': TOKEN_TYPE.DOT,
        '?': TOKEN_TYPE.QUESTION_MARK,
        '(': TOKEN_TYPE.OPEN_PARENTHESIS,
        ')': TOKEN_TYPE.CLOSED_PARENTHESIS,
        '[': TOKEN_TYPE.OPEN_SQUARE_BRACKET,
        ']': TOKEN_TYPE.CLOSED_SQUARE_BRACKET,
        '-': TOKEN_TYPE.DASH
    }
    for char in regex_string:
        if char in meta_chars:
            tokens.append(token(meta_chars[char], char))  
        else:
            tokens.append(token(TOKEN_TYPE.LITERAL_CHARACTER, char))
    return tokens
print(tokenize("a|b*"))

In [ ]:
# Abstract Syntax Tree (AST) Nodes
class AST_Node:
    pass

class Literal_Node(AST_Node):
    def __init__(self, char: chr):
        self.char = char
    def __repr__(self):
        return f"Literal('{self.char}')"

class Or_Node(AST_Node):
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def __repr__(self):
        return f"Or({self.left}, {self.right})"

class Seq_Node(AST_Node):
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def __repr__(self):
        return f"Seq({self.left}, {self.right})"

class Star_Node(AST_Node):
    def __init__(self, child):
        self.child = child
    def __repr__(self):
        return f"Star({self.child})"

class Plus_Node(AST_Node):
    def __init__(self, child):
        self.child = child
    def __repr__(self):
        return f"Plus({self.child})"

class Optional_Node(AST_Node):
    def __init__(self, child):
        self.child = child
    def __repr__(self):
        return f"Optional({self.child})"

class Character_Class_Node(AST_Node):
    def __init__(self, chars: set):
        self.chars = chars
    def __repr__(self): 
        return f"CharClass({sorted(list(self.chars))})"

In [ ]:
# Parser Grammer Rules
#  regex-parse, parse-or, parse-seq, parse-quantified, parse-base, parse-square-bracket, parse-square-bracket
class Parser:
    def __init__(self, tokens : list[token]):
        self.tokens = tokens
        self.pos = 0

    def current_token(self) -> token:
        if self.pos < len(self.tokens):
            return self.tokens[self.pos]
        return None

    def advance(self):
        self.pos += 1

    #   regex-expr ::= regex-or-expr
    #   Tokens -> AST
    def regex_parse(self):
        return self.parse_or()

    #regex-or-expr ::= regex-seq-expr (OR_TOKEN regex-seq-expr)*
    def parse_or(self):
        result = self.parse_seq()

        if self.current_token() is None:
            return result

        while self.current_token() is not None and self.current_token().token_type is TOKEN_TYPE.OR:
            self.advance()

            right = self.parse_seq()
            result = Or_Node(left=result, right=right)

        return result

    #regex-seq-expr ::= regex-quantified-expr (regex-quantified-expr)*
    def parse_seq(self):
        result = self.parse_quantified()

        if self.current_token() is None:
            return result

        while self.current_token() is not None and self.current_token().token_type not in {TOKEN_TYPE.OR, TOKEN_TYPE.CLOSED_PARENTHESIS}:
            right = self.parse_quantified()
            result = Seq_Node(left=result, right=right)
        
        return result

#regex-quantified-expr ::= regex-base-expr (STAR_TOKEN | PLUS_TOKEN |      
#                          QUESTION_MARK_TOKEN)?
    def parse_quantified(self):
        left = self.parse_base()
        
        if self.current_token() is None:
            return left

        curr = self.current_token().token_type
        if curr in {TOKEN_TYPE.STAR, TOKEN_TYPE.PLUS, TOKEN_TYPE.QUESTION_MARK}:
            self.advance()

        if curr is TOKEN_TYPE.STAR:
            return Star_Node(left)
        elif curr is TOKEN_TYPE.PLUS:
            return Plus_Node(left)
        elif curr is TOKEN_TYPE.QUESTION_MARK:
            return Optional_Node(left)

        return left

#	regex-base-expr ::= LITERAL_CHAR_TOKEN
#                       |   OPEN_SQUARE_BRACKET_TOKEN            
#                           square-bracket-content                                 
#                           CLOSED_SQUARE_BRACKET_TOKEN
#			            |   OPEN_PARENTHESIS_TOKEN 
#                           regex-expr 
#                           CLOSED_PARENTHESIS_TOKEN
    def parse_base(self):
        curr = self.current_token()
        if not curr:
            raise ValueError("Unexpected end of input")
            
        self.advance()

        if curr.token_type is TOKEN_TYPE.LITERAL_CHARACTER:
            return Literal_Node(curr.value)
        
        elif curr.token_type is TOKEN_TYPE.OPEN_SQUARE_BRACKET:
            return self.parse_square_bracket()
        
        elif curr.token_type is TOKEN_TYPE.OPEN_PARENTHESIS:
            expr = self.regex_parse()
            
            # make sure that the ( is closed
            if self.current_token() is not None and self.current_token().token_type is TOKEN_TYPE.CLOSED_PARENTHESIS:
                self.advance()
                return expr
            else:
                raise ValueError("Missing closing parenthesis ')'")
                
        else:
            raise ValueError(f"Unexpected token in base: {curr.token_type}")
        
#	square-bracket-content ::= square-bracket-element+
#   square-bracket-element ::= LITERAL_CHAR_TOKEN 
#				             | LITERAL_CHAR_TOKEN DASH LITERAL_CHAR_TOKEN
    def parse_square_bracket(self):
        valid_chars = set()
        
        while self.current_token() is not None and self.current_token().token_type is not TOKEN_TYPE.CLOSED_SQUARE_BRACKET:
            first_char = self.current_token().value
            self.advance()

            if self.current_token() is not None and self.current_token().token_type is TOKEN_TYPE.DASH:
                self.advance() # consume "-"

                last_char = self.current_token().value
                self.advance() # consume last char

                for char in range(ord(first_char), ord(last_char) + 1):
                    valid_chars.add(chr(char))
            else:
                valid_chars.add(first_char)
        
        if self.current_token() is None or self.current_token().token_type is not TOKEN_TYPE.CLOSED_SQUARE_BRACKET:
            raise ValueError("missing ']'")
            
        self.advance() # consume ']
        
        return Character_Class_Node(valid_chars)


In [ ]:
# Example Usage
tokens = tokenize("abc[x-z]*(t|y)?")

parser = Parser(tokens=tokens)

print(parser.regex_parse())

In [ ]:
# State and NFA classes
state_counter = 0

def create_state():
    global state_counter
    new_state = State(f"S{state_counter}")
    state_counter += 1
    return new_state

class State():
    def __init__(self, name):
        self.name = name
        self.transitions: dict[str, str | list[str]]= {} # S0: S1 or epsilon: [S0, S1, etc] 
        self.is_terminating: bool = False

    def add_transition(self, char: str, to_state: 'State'):
        target_name = to_state.name

        if char not in self.transitions:
            # "A": "S0" -> not a list
            self.transitions[char] = target_name
        else:
            # if it exists make it a list
            # "epsilon": "S0" -> ["S0", "S1"]
            if isinstance(self.transitions[char], str):
                old = self.transitions[char]
                self.transitions[char] = [old, target_name]

            elif isinstance(self.transitions[char], list):
                self.transitions[char].append(target_name)

class NFA():
    def __init__(self, start_state: State, accept_state: State, states:set[State]=None):
        self.start = start_state
        self.accept = accept_state

        if states is None:
            self.states = {start_state, accept_state}
        else:
            self.states = states

In [ ]:
# NFA Construction from AST
EPSILON = "\u03b5"

def AST_to_NFA(node: AST_Node) -> NFA:
    if isinstance(node, Literal_Node):
        # if Char then, start --char--> end
        start = create_state()
        end = create_state()

        start.add_transition(char=node.char, to_state=end)

        return NFA(start_state= start, accept_state=end)
    
    elif isinstance(node, Character_Class_Node):
        # if chars, start -a-> end & start -b-> end & etc
        start = create_state()
        end = create_state()
        
        for char in node.chars:
            start.add_transition(char=char, to_state=end)

        return NFA(start_state=start, accept_state=end)
    
    elif isinstance(node, Or_Node):
        left = AST_to_NFA(node.left)
        right = AST_to_NFA(node.right)

        start = create_state()
        accept = create_state()

        start.add_transition(EPSILON, left.start)
        start.add_transition(EPSILON, right.start)

        left.accept.add_transition(EPSILON, accept)
        right.accept.add_transition(EPSILON, accept)

        combined_states = left.states | right.states | {start, accept}

        return NFA(start, accept, states=combined_states)

    elif isinstance(node, Seq_Node):
        left = AST_to_NFA(node.left)
        right = AST_to_NFA(node.right)

        left.accept.add_transition(EPSILON, right.start)

        combined_states = left.states | right.states

        return NFA(start_state=left.start, accept_state=right.accept, states=combined_states)

    elif isinstance(node, (Star_Node, Plus_Node)):
        nfa = AST_to_NFA(node=node.child)

        start = create_state()
        accept = create_state()

        # from start to left is epsilon transition)
        start.add_transition(EPSILON, nfa.start)
        # only add the skip if its *
        if isinstance(node, Star_Node):
            start.add_transition(EPSILON, accept)

        nfa.accept.add_transition(EPSILON, accept)
        nfa.accept.add_transition(EPSILON, nfa.start)

        combined_states = nfa.states | {start, accept}

        return NFA(start, accept, combined_states)

    elif isinstance(node, Optional_Node):
        nfa = AST_to_NFA(node.child)

        start = create_state()
        accept = create_state()

        # normal start
        start.add_transition(EPSILON, nfa.start)
        # can skip
        start.add_transition(EPSILON, accept)

        # normal end
        nfa.accept.add_transition(EPSILON, accept)

        combined_states = nfa.states | {start, accept}
        return NFA(start, accept, combined_states)

In [ ]:
# Exporting NFA to JSON and Graph
import json
import graphviz

def export_nfa_to_json(final_nfa, filename="nfa_output.json"):
    final_nfa.accept.is_terminating = True
    output_dict = {"startingState": final_nfa.start.name}
    
    for state in final_nfa.states:
        state_info = {"isTerminatingState": state.is_terminating}
        for char, targets in state.transitions.items():
            state_info[char] = targets
        output_dict[state.name] = state_info
        
    json_string = json.dumps(output_dict, indent=4, ensure_ascii=False)
    with open(filename, "w", encoding="utf-8") as f:
        f.write(json_string)
    return json_string


def draw_nfa(json_string, filename = "nfa_output_graph"):
    nfa_data = json.loads(json_string)
    
    dot = graphviz.Digraph(name='NFA', format='png')
    dot.attr(rankdir='LR') 
    
    starting_state = nfa_data.pop("startingState")
    
    dot.node('fake_start', shape='point')
    dot.edge('fake_start', starting_state)
    
    for state, info in nfa_data.items():
        node_shape = 'doublecircle' if info.get("isTerminatingState") else 'circle'
        dot.node(state, shape=node_shape)
            
        for char, targets in info.items():
            if char == "isTerminatingState":
                continue
                
            if isinstance(targets, str):
                targets = [targets]
                
            for target in targets:
                dot.edge(state, target, label=char)
                
    dot.render(filename, view=False, cleanup=True)
    return dot

###########################################################################
test_regex = "(0|1(01*0)*1)*" # "([a-c]+|x?)*(y|z)" #"(a|b)*a(a|b)(a|b)"

# lexer
tokens = tokenize(test_regex)
# parse tokens into AST
ast_root = Parser(tokens).regex_parse()

# Thompson's algo
global state_counter 
state_counter = 0 
final_nfa = AST_to_NFA(ast_root)

# Output
json_output = export_nfa_to_json(final_nfa)
graph = draw_nfa(json_output)

print("check nfa_output.json")